# Pandas Chunk Loading – Memory Consumption

This helps understand **why `chunksize` is useful** when loading large datasets.

We will demonstrate:
1. Creating a large dataset
2. Loading the dataset normally
3. Measuring memory usage
4. Loading the dataset using chunks
5. Observing how memory stays low


## Step 1: Import Libraries

In [10]:
import pandas as pd
import numpy as np
import os
import psutil

## Step 2: Function to Check RAM Usage

We use `psutil` to check how much RAM the Python process is consuming.

In [11]:
process = psutil.Process(os.getpid())

def check_memory():
    mem = process.memory_info().rss / (1024 * 1024)
    print(f"Current RAM Usage: {mem:.2f} MB")

In [12]:
check_memory()

Current RAM Usage: 25.31 MB


## Step 3: Create a Large Dataset
We simulate a dataset with **1 million rows**.

In [13]:
rows = 1_000_000

data = pd.DataFrame({
    'age': np.random.randint(18, 60, rows),
    'salary': np.random.randint(20000, 120000, rows),
    'experience': np.random.randint(0, 20, rows)
})

data.to_csv('big_dataset.csv', index=False)

print('Dataset shape:', data.shape)

Dataset shape: (1000000, 3)


## Step 4: Check Memory Before Loading

In [14]:
check_memory()

Current RAM Usage: 43.43 MB


## Step 5: Load Entire Dataset Normally

Here pandas loads **all rows into memory at once**.

In [15]:
df = pd.read_csv('big_dataset.csv')
print(df.shape)

check_memory()

(1000000, 3)
Current RAM Usage: 68.11 MB


Observation:

- RAM usage increases because the **entire dataset is stored in memory**.

## Step 6: Delete DataFrame to Free Memory

In [16]:
del df

check_memory()

Current RAM Usage: 45.36 MB


## Step 7: Load Dataset Using Chunks

Now pandas loads **only small pieces of the dataset at a time**.

In [17]:
total_rows = 0

for chunk in pd.read_csv('big_dataset.csv', chunksize=10000):
    total_rows += chunk.shape[0]
    check_memory()

print('Total rows processed:', total_rows)

Current RAM Usage: 45.90 MB
Current RAM Usage: 46.21 MB
Current RAM Usage: 46.35 MB
Current RAM Usage: 46.97 MB
Current RAM Usage: 46.97 MB
Current RAM Usage: 46.97 MB
Current RAM Usage: 46.98 MB
Current RAM Usage: 47.59 MB
Current RAM Usage: 47.60 MB
Current RAM Usage: 47.61 MB
Current RAM Usage: 46.42 MB
Current RAM Usage: 47.04 MB
Current RAM Usage: 47.19 MB
Current RAM Usage: 47.19 MB
Current RAM Usage: 47.19 MB
Current RAM Usage: 47.19 MB
Current RAM Usage: 47.72 MB
Current RAM Usage: 46.57 MB
Current RAM Usage: 46.58 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 47.22 MB
Current RAM Usage: 4

In [18]:
check_memory()

Current RAM Usage: 44.30 MB


## Final Observation

When using `chunksize`:

- Only **10,000 rows are loaded at a time**
- RAM usage remains **almost constant**
- This allows processing **very large datasets (GBs or TBs)**